# Layer: Bronze (Raw/Ingestion)
**Project:** Lean Logistics Data Pipeline  
**Business Domain:** E-commerce (Olist Dataset)

---
## 📑 Notebook Information
| Version | Date | Author | Summary of Changes |
| :--- | :--- | :--- | :--- |
| v1.0 | 2026-02-20 | Tássia Marchito | Initial ingestion of CSV files to Delta tables. |
| v1.1 | 2026-02-20 | Tássia Marchito | Fixed UC `_metadata` error and added dynamic file discovery loop. |
| v1.2 | 2026-02-22 | Tássia Marchito | Refactored metadata to English and implemented standard audit columns (`ts_ingestion`). |

---
## 🎯 Objectives
The Bronze layer acts as the entry point for all raw data. It preserves the state of the source system while adding governance.
* **Fidelity:** Data is stored exactly as it arrived from the source (CSV).
* **Immutability:** Historical records are preserved with ingestion timestamps.
* **Traceability:** Every row includes the source file path via Unity Catalog `_metadata`.
* **Governance:** Initial table registration in the Catalog with basic tags and primary key definitions.

In [0]:
%skip
display(dbutils.fs.ls("/Volumes/cat_tm_services_bronze/db_logistics/raw_files"))

In [0]:
%skip
%sql
-- 1. Removendo tabelas sem o prefixo tb_ no novo schema
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.customers;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.geolocation;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.order_items;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.order_payments;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.order_reviews;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.orders;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.products;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.sellers;
DROP TABLE IF EXISTS cat_tm_services_bronze.db_logistics.product_category_name_translation;

-- 2. (OPCIONAL) Removendo o schema antigo 'logistics' caso ele ainda exista
-- ATENÇÃO: Use CASCADE para apagar tudo que estiver dentro dele
-- DROP SCHEMA IF EXISTS cat_tm_services_bronze.logistics CASCADE;

In [0]:
from pyspark.sql.functions import current_timestamp, col
import os

In [0]:
import os
from pyspark.sql.functions import current_timestamp

# 1. Path and Destination Configuration
target_schema = "cat_tm_services_bronze.db_logistics"
volume_path = "/Volumes/cat_tm_services_bronze/db_logistics/raw_files"

# 2. Metadata Dictionary (Including PKs for Silver Layer Lineage)
bronze_metadata = {
    "olist_customers_dataset.csv": {
        "pk": "customer_id",
        "area": "crm", 
        "cm": "Customer information and location."
    },
    "olist_geolocation_dataset.csv": {
        "pk": "geolocation_zip_code_prefix, geolocation_lat, geolocation_lng", 
        "area": "logistics", 
        "cm": "Geographic coordinates for zip codes."
    },
    "olist_order_items_dataset.csv": {
        "pk": "order_id, order_item_id", 
        "area": "sales", 
        "cm": "Items contained in each order."
    },
    "olist_order_payments_dataset.csv": {
        "pk": "order_id, payment_sequential", 
        "area": "finance", 
        "cm": "Payment methods and installment details."
    },
    "olist_order_reviews_dataset.csv": {
        "pk": "review_id", 
        "area": "crm", 
        "cm": "Customer satisfaction reviews."
    },
    "olist_orders_dataset.csv": {
        "pk": "order_id", 
        "area": "sales", 
        "cm": "Core order management data."
    },
    "olist_products_dataset.csv": {
        "pk": "product_id", 
        "area": "catalog", 
        "cm": "Product catalog details."
    },
    "olist_sellers_dataset.csv": {
        "pk": "seller_id", 
        "area": "partners", 
        "cm": "Sellers information."
    },
    "product_category_name_translation.csv": {
        "pk": "product_category_name", 
        "area": "catalog", 
        "cm": "Translation of category names to English."
    }
}

# 3. Dynamic File Discovery
files_in_volume = [f.name for f in dbutils.fs.ls(volume_path) if f.name.endswith(".csv")]

print(f"📦 Starting High-Performance Bronze Ingestion for {len(files_in_volume)} files...\n")

for file_name in files_in_volume:
    # Name Standardization: olist_orders_dataset.csv -> tb_orders
    clean_name = file_name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    table_name = f"tb_{clean_name}"
    full_target_path = f"{target_schema}.{table_name}"
    full_source_path = os.path.join(volume_path, file_name)
    
    # Fetch metadata or use defaults
    info = bronze_metadata.get(file_name, {"area": "general", "cm": "Raw ingested data."})
    
    print(f"🔄 Processing: {table_name}")
    
    # OPTIMIZED READ: inferSchema=False prevents double scanning of the source files
    # Data is ingested as strings to ensure maximum speed and reliability in the Bronze layer.
    df_raw = (spark.read.format("csv")
              .option("header", "true")
              .option("inferSchema", "false")  # Performance Boost: avoid double scan
              .option("sep", ",")              # Explicitly defined separator
              .load(full_source_path)
              .select("*", "_metadata.file_path")
              .withColumnRenamed("file_path", "_source_file")
              .withColumn("ts_ingestion", current_timestamp())) # Lineage & Audit
    
    # WRITE: Delta Format with Schema Overwrite enabled for flexibility
    (df_raw.write.format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(full_target_path))
    
    # GOVERNANCE: Apply metadata and tags directly to Unity Catalog
    spark.sql(f"COMMENT ON TABLE {full_target_path} IS '{info['cm']}'")
    spark.sql(f"ALTER TABLE {full_target_path} SET TAGS ('quality' = 'bronze', 'domain' = '{info['area']}', 'type' = 'raw')")
    
    print(f"✅ Table {table_name} processed successfully.\n")

print("🚀 High-Performance Bronze Ingestion Complete!")

In [0]:
# --- Bronze Quality Assurance (QA Gate) ---
# Objective: Audit the ingestion process by comparing Row Counts and Schema Consistency.

from pyspark.sql.functions import count

print(f"🧐 Auditing Bronze Ingested Assets...")

# 1. Verification of Table Creation vs Metadata Expectation
expected_tables = [f"tb_{f.replace('olist_', '').replace('_dataset.csv', '').replace('.csv', '')}" for f in bronze_metadata.keys()]
actual_tables = [t.name for t in spark.catalog.listTables(target_schema)]

missing_tables = set(expected_tables) - set(actual_tables)

# 2. Detailed Row Count Reconciliation
print(f"{'Table Name':<30} | {'Source Count':<12} | {'Bronze Count':<12} | {'Status'}")
print("-" * 75)

pass_count = 0
for file_name, meta in bronze_metadata.items():
    clean_name = file_name.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    t_name = f"tb_{clean_name}"
    
    try:
        # Get counts
        source_path = os.path.join(volume_path, file_name)
        source_cnt = spark.read.format("csv").option("header", "true").load(source_path).count()
        bronze_cnt = spark.table(f"{target_schema}.{t_name}").count()
        
        # Check integrity
        status = "✅ MATCH" if source_cnt == bronze_cnt else "❌ MISMATCH"
        if status == "✅ MATCH": pass_count += 1
        
        print(f"{t_name:<30} | {source_cnt:<12} | {bronze_cnt:<12} | {status}")
        
    except Exception as e:
        print(f"{t_name:<30} | Error accessing data: {e}")

# 3. Final Ingestion Report
print("-" * 75)
print(f"📊 Final Report: {pass_count}/{len(bronze_metadata)} tables successfully reconciled.")

if len(missing_tables) == 0 and pass_count == len(bronze_metadata):
    print("\n✅ QA STATUS: SUCCESS")
    print("Full-fidelity ingestion verified. Bronze layer is ready for Silver refinement.")
else:
    if missing_tables:
        print(f"\n⚠️ QA STATUS: WARNING - Missing Tables: {missing_tables}")
    else:
        print("\n❌ QA STATUS: FAILED - Data Loss detected during ingestion.")